# Kerr ell=4 decoupled hybrid -- reproduce `phase_c_l4_decoupled` (2026-07-04)

Reproduces the exact model documented in `kerr/configs/hybrid_kerr_l4_decoupled.yaml`
(k8 order-2 prior in, order-4 Richardson target, ell=4, mode-selective m1-m5 consensus
ensemble at scri). Drive-backed and resumable: every corpus split and the trained model
are saved to `MyDrive/kerr_l4/`, so a disconnect never loses work.

**PILOT vs FULL** is one switch in Cell 5:
- pilot (default): `N_TRAIN,N_VAL,N_TEST = 128,64,64` -- a real but small run for a preview
  and a go/no-go decision (~1 h; corpus build is the long pole, ~100 s/sample CPU).
- full (exact reproduction): set `128,64,64 -> 1024,128,128` and re-run from Cell 6.

**Parallel variants (shared corpus):** Cell 5 has a `VARIANT` switch (`baseline` = scalar norm, the exact repro; `envelope` = per-time envelope norm; `gate` = amplitude-gated residual -- both tau fixes). All variants read the SAME corpus and write to a distinct `out_dir`/Drive folder, so you can run them in **parallel sessions**: build the corpus ONCE (run Cell 6 in the first session; wait for `saved to Drive`), then in each other session set `VARIANT`, run Cells 2-5, Cell 6 (reuses corpus from Drive, no rebuild), and Cell 7. Each session's results land in its own Drive folder.

**GPU:** use G4 (96 GB) -- training runs at the config `batch_size: 8` on the 881x801 grid
and needs > 40 GB (the analogous SW t=100 hybrid peaked ~76 GB). On A100 (40 GB) it will
OOM at batch 8; only then set `train.batch_size: 4` in the config (preview only, not exact).

**Re-eval without retraining (Cell 7b):** after refreshing the repo code (re-running
Cell 4), you do NOT need to retrain. Run Cell 6 (restores the corpus from Drive), then
**Cell 7b** -- it restores the trained `model.pt` from Drive and re-runs extraction +
plots with the current code (~1-2 min for the full run). Re-running Cell 4 is safe: the
corpus and model live on Drive and are restored by Cell 6 / Cell 7b.

**Cell 6 sanity-checks the corpus** (finite fields, ell=4 Kerr Mw ~[0.8,1.03]) before training.
**Cell 8 prints the decision table**; **Cell 9 shows all 7 figures**. Run top-to-bottom.

In [ ]:
# Cell 2 -- mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/kerr_l4'
os.makedirs(DRIVE, exist_ok=True)
print('Drive work dir:', DRIVE)
!ls -lh {DRIVE} 2>/dev/null || echo '(empty - first run)'

In [ ]:
# Cell 3 -- sanity: GPU / CPU / RAM  (corpus build is CPU-bound; more vCPUs = faster)
!nvidia-smi
!nproc
!free -g | head -2

In [ ]:
# Cell 4 -- clone + pinned deps (neuralop + qnm) + allocator flag + qnm dependency check
import os, sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['MPLBACKEND'] = 'Agg'
%cd /content
!rm -rf QNM_Project_Fresh
!git clone --depth 1 https://github.com/ScreaM835/QNM_Project_Fresh.git
REPO = '/content/QNM_Project_Fresh'
%cd {REPO}
# qnm (Leo Stein) supplies the Kerr Leaver reference the build + eval need -- NOT a Colab default.
!pip -q install "neuraloperator==2.0.0" "qnm==0.4.4"
sys.path.insert(0, REPO)
import torch; print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))
import qnm
try:
    from kerr.src.qnm_kerr_reference import kerr_qnm
    print('qnm Kerr ref OK: Mw(a=0.9, l=4) =', round(kerr_qnm(0.9, 4, 2, 0).M_omega_R, 4))
except Exception as e:
    print('warming qnm data ...', repr(e)); qnm.download_data()
    from kerr.src.qnm_kerr_reference import kerr_qnm
    print('qnm Kerr ref OK after download: Mw =', round(kerr_qnm(0.9, 4, 2, 0).M_omega_R, 4))

In [ ]:
# Cell 5 -- config: PILOT vs FULL + VARIANT (shared corpus) + paths
# pilot = a real but small run (preview + decision). FULL exact repro = 1024,128,128.
N_TRAIN, N_VAL, N_TEST = 128, 64, 64      # <-- set to 1024, 128, 128 for the exact reproduction
WORKERS = os.cpu_count() or 8             # corpus build parallelism (Linux fork)

# ---- VARIANT: change this ONE line per parallel session ------------------
# Every variant SHARES the same corpus (same data.dir) and writes to a DISTINCT
# out_dir, so you can run them in parallel notebooks/sessions without collision.
# Build the corpus ONCE (Cell 6 in the FIRST session); the others reuse it from Drive.
VARIANT = 'baseline'                      # 'baseline' | 'envelope' | 'gate'
VARIANTS = {
    'baseline': ('kerr/configs/hybrid_kerr_l4_decoupled.yaml',      'fno_hybrid_kerr'),
    'envelope': ('kerr/configs/hybrid_kerr_l4_decoupled_env.yaml',  'fno_hybrid_kerr_env'),
    'gate':     ('kerr/configs/hybrid_kerr_l4_decoupled_gate.yaml', 'fno_hybrid_kerr_gate'),
}
CONFIG, RUN_NAME = VARIANTS[VARIANT]
# --------------------------------------------------------------------------

REPO = '/content/QNM_Project_Fresh'
DRIVE = '/content/drive/MyDrive/kerr_l4'
DATA = f'{REPO}/kerr/outputs/phase_c_l4_decoupled'     # SHARED corpus dir (all variants)
OUT  = f'{DATA}/{RUN_NAME}'                            # per-variant model + eval dir
DRIVE_OUT = f'{DRIVE}/{RUN_NAME}'                      # per-variant Drive backup
os.makedirs(DATA, exist_ok=True)
print(f'VARIANT={VARIANT}  config={CONFIG}')
print(f'  out_dir  = {OUT}')
print(f'  drive_out= {DRIVE_OUT}')
print(f'PILOT sizes: train {N_TRAIN}, val {N_VAL}, test {N_TEST}  (full = 1024/128/128)')
print(f'workers={WORKERS}  est. corpus build ~{(N_TRAIN+N_VAL+N_TEST)*100/max(WORKERS,1)/60:.0f} min (~100 s/sample)')


In [ ]:
# Cell 6 -- corpus: reuse each split from Drive if present, else build (subprocess = robust
# quoting, no fragile shell line-continuation) and SAVE to Drive. Then sanity-check.
import os, sys, shutil, time, subprocess
import numpy as np
if REPO not in sys.path:
    sys.path.insert(0, REPO)

def build_split(S, N):
    cmd = ['python', 'kerr/scripts/build_kerr_dataset_lscan.py', '--ell', '4',
           '--split', S, '--coarse-n', '2:401,4:201,8:101', '--ks', '2', '4', '8',
           '--grid-order', '1:4,2:4,4:4,8:2', '--n', str(N),
           '--out', f'{DATA}/dataset_{S}.npz', '--workers', str(WORKERS)]
    if subprocess.run(cmd, cwd=REPO).returncode != 0:
        raise SystemExit(f'[corpus] BUILD FAILED for {S}')

for S, N in [('train', N_TRAIN), ('val', N_VAL), ('test', N_TEST)]:
    repo_npz  = f'{DATA}/dataset_{S}.npz'
    drive_npz = f'{DRIVE}/dataset_{S}_n{N}.npz'
    if os.path.exists(drive_npz):
        print(f'[corpus] reuse {S} (n={N}) from Drive'); shutil.copy(drive_npz, repo_npz)
    else:
        print(f'[corpus] building {S} n={N} ...', flush=True); t0 = time.time()
        build_split(S, N)
        shutil.copy(repo_npz, drive_npz)
        print(f'[corpus] {S} built + saved to Drive in {(time.time()-t0)/60:.1f} min')

# --- sanity check (pilot-safe: does NOT require n == the full 1024/128/128) ---
from kerr.src.kerr_dataset import load_dataset
print('\n=== corpus sanity ===')
ok = True
for S in ['train', 'val', 'test']:
    _, arr, grids, _ = load_dataset(f'{DATA}/dataset_{S}.npz')
    fin = bool(np.all(np.isfinite(arr['psi_fine_re'])) and np.all(np.isfinite(arr['psi_fine_im'])))
    aM, mw = arr['P'][:, 0], arr['qnm'][:, 0]
    ok = ok and fin and (0.78 < mw.min()) and (mw.max() < 1.20)
    print('  {:5s}: n={} ntau={} fine={} finite={} a/M[{:.2f},{:.2f}] Mw[{:.3f},{:.3f}]'.format(
        S, arr['P'].shape[0], grids.tau.size, arr['psi_fine_re'].shape, fin,
        aM.min(), aM.max(), mw.min(), mw.max()))
print('  SANITY', 'PASS (fields finite, ell=4 Kerr Mw in range)' if ok else 'FAIL -- STOP, do NOT train')
assert ok, 'corpus sanity failed -- inspect the build output before training'

In [ ]:
# Cell 7 -- train + eval + plots (one script). Saves the whole out_dir to Drive.
# Runs the VARIANT selected in Cell 5. To re-eval a model ALREADY on Drive without
# retraining, SKIP this cell and run Cell 7b instead.
import os, shutil
%cd {REPO}
!MPLBACKEND=Agg PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python kerr/scripts/train_eval_hybrid_kerr.py --config {CONFIG}
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copytree(OUT, DRIVE_OUT, dirs_exist_ok=True)
print('[train] out_dir saved -> Drive:', DRIVE_OUT)


In [ ]:
# Cell 7b -- RE-EVAL ONLY (no retrain). Restore the trained model for the current
# VARIANT from Drive and re-run extraction + plots with the CURRENT repo code. Use
# this after a code refresh (re-running Cell 4) instead of re-training -- the model
# on Drive is reused as-is. Requires the corpus restored first (run Cell 6).
# For the FULL run this is ~1-2 min vs ~30-90 min to retrain.
import os, shutil
os.makedirs(OUT, exist_ok=True)
restored = []
for f in ['model.pt', 'history.json']:
    src = f'{DRIVE_OUT}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{OUT}/{f}'); restored.append(f)
    else:
        print('WARNING: missing on Drive ->', src)
assert 'model.pt' in restored, f'no model.pt in {DRIVE_OUT} -- run Cell 7 (train) for VARIANT={VARIANT} first'
print('[eval-only] restored from Drive:', restored)
%cd {REPO}
!MPLBACKEND=Agg PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python kerr/scripts/train_eval_hybrid_kerr.py --config {CONFIG} --eval-only
shutil.copytree(OUT, DRIVE_OUT, dirs_exist_ok=True)
print('[eval-only] refreshed report/per_sample/figs -> Drive:', DRIVE_OUT)


In [ ]:
# Cell 8 -- *** DECISION TABLE: consensus AND best-of-suite ***
import json
d = json.load(open(f'{OUT}/report.json'))
fr = d['field_rl2']
mw, ta = d['qnm_Mw_err_pct'], d['qnm_tau_err_pct']
mwb, tab = d['qnm_Mw_err_pct_best'], d['qnm_tau_err_pct_best']
sp, sd, g = d['qnm_spread_pct'], d['qnm_sel_dist'], d['acceptance_gate']
print('=' * 80)
print(f"  KERR ell=4 DECOUPLED HYBRID  (n_test={d['n_test']})   [medians]")
print('=' * 80)
print('OVERALL (prior -> hybrid, fine = extraction floor):')
print('  field rel-L2        : {:6.2f}% -> {:6.3f}%   (richardson {:.3f}%)'.format(
    fr['prior']['median']*100, fr['hybrid']['median']*100, fr['richardson']['median']*100))
print('  Mw  err (consensus) : {:6.2f}% -> {:6.3f}%   (fine {:.3f}%)'.format(
    mw['prior']['median'], mw['hybrid']['median'], mw['fine']['median']))
print('  Mw  err (best-suite): {:6.2f}% -> {:6.3f}%   (fine {:.3f}%)'.format(
    mwb['prior']['median'], mwb['hybrid']['median'], mwb['fine']['median']))
print('  tau err (consensus) : {:6.2f}% -> {:6.3f}%   (fine {:.3f}%)'.format(
    ta['prior']['median'], ta['hybrid']['median'], ta['fine']['median']))
print('  tau err (best-suite): {:6.2f}% -> {:6.3f}%   (fine {:.3f}%)'.format(
    tab['prior']['median'], tab['hybrid']['median'], tab['fine']['median']))
print('  ensemble spread (hybrid) {:.2f}% | sel_dist (hybrid) {:.4f}'.format(
    sp['hybrid']['median'], sd['hybrid']['median']))
print('-' * 80)
print('BY SPIN, best-of-suite (prior -> hybrid [fine]):')
hdr = '{:>12} | {:>2} | {:>16} | {:>18} | {:>18}'
print(hdr.format('a/M bin', 'n', 'field %', 'Mw % (best)', 'tau % (best)'))
def f3(x):
    return '  nan' if x is None or x != x else '{:.2f}'.format(x)
for b in d['by_spin']:
    print(hdr.format(
        '[{:.2f},{:.2f}]'.format(*b['bin']), b['n'],
        '{:.1f}->{:.2f}'.format(b['field_rl2_prior']*100, b['field_rl2_hybrid']*100),
        '{}->{} [{}]'.format(f3(b.get('qnm_Mw_err_prior_best')), f3(b.get('qnm_Mw_err_hybrid_best')), f3(b.get('qnm_Mw_err_fine_best'))),
        '{}->{} [{}]'.format(f3(b.get('qnm_tau_err_prior_best')), f3(b.get('qnm_tau_err_hybrid_best')), f3(b.get('qnm_tau_err_fine_best')))))
print('-' * 80)
print('best-of-suite = per-sample Leaver-closest extractor (extractors are Leaver-blind);')
print('consensus = robust ensemble median. per_sample.json has the full per-method table.')
print('gate (arbitrary thresholds): field_pass={} qnm_pass={} OVERALL={}'.format(
    g['field_pass'], g['qnm_pass'], g['overall_pass']))
print('=' * 80)

In [ ]:
# Cell 9 -- show all 7 figures inline
from IPython.display import Image, display
import os
figs = f'{OUT}/figs'
for fn in ['hybrid_field_vs_spin.png', 'hybrid_qnm_vs_spin.png', 'hybrid_tau_vs_spin.png',
           'hybrid_ringdown_scri.png', 'hybrid_pointwise_error_hybrid.png',
           'hybrid_pointwise_error_baseline.png', 'hybrid_loss.png']:
    p = os.path.join(figs, fn)
    if os.path.exists(p):
        print(fn); display(Image(p))

In [ ]:
# Cell 10 -- zip report + per_sample + figures and download to the laptop
import os
!cd {OUT} && zip -r /content/kerr_l4_pilot.zip report.json per_sample.json figs history.json
from google.colab import files
files.download('/content/kerr_l4_pilot.zip')